# Test isolé de l’extraction des pages structurées

Ce notebook compare les deux stratégies sur les six pages problématiques. Il ne modifie aucun fichier du corpus.


In [1]:
from pathlib import Path
import json, sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from config import load_config
from cleaner import HTMLCleaner
from scraper import url_hash

cfg = load_config(ROOT / 'config.yaml')
cleaner = HTMLCleaner(cfg)
print(f'Projet : {ROOT}')


Projet : <PROJECT_ROOT>


In [2]:
TARGETS = {
    'master_management': 'https://ensea.ed.ci/formation-initiale/master-en-management/',
    'doctorants': 'https://ensea.ed.ci/formation-doctorale/doctorants/',
    'seminaires': 'https://ensea360.ensea.ed.ci/formation-continue/seminaires-de-recherche/',
    'partenaires_recherche': 'https://ensea.ed.ci/cooperation/partenaires-de-recherche/',
    'couts': 'https://ensea.ed.ci/formation-initiale/couts/',
    'tarifs': 'https://ensea.ed.ci/faq-items/quels-sont-les-tarifs/',
}

documents = {}
rows = []
for name, url in TARGETS.items():
    raw_path = cfg.project.data_path / 'raw' / f'{url_hash(url)}.json'
    record = json.loads(raw_path.read_text(encoding='utf-8'))
    document = cleaner._clean_single(record)
    documents[name] = document
    diagnostics = document.extraction_diagnostics
    rows.append({
        'page': name,
        'méthode': document.extraction_method,
        'trafilatura_chars': diagnostics.get('trafilatura_chars'),
        'structured_main_chars': diagnostics.get('structured_main_chars'),
        'selected_chars': diagnostics.get('selected_chars'),
    })
pd.DataFrame(rows)


,page,méthode,trafilatura_chars,structured_main_chars,selected_chars
0,master_management,trafilatura_precision,2512,2533,2512
1,doctorants,structured_main_richer,151,5833,5833
2,seminaires,trafilatura_precision,157,23,157
3,partenaires_recherche,structured_main_richer,547,5882,5882
4,couts,trafilatura_precision,829,989,829
5,tarifs,trafilatura_precision,269,293,269


In [3]:
for name in ('doctorants', 'partenaires_recherche'):
    print('\n' + '=' * 100)
    print(name, '—', documents[name].extraction_method)
    print(documents[name].markdown[:1500])



doctorants — structured_main_richer
Doctorants
All
2019
2020
2021
2022
CISEA-2023
FE Doukouré Charles
FE Doukouré Charles
FE Doukouré Charles
Impact de la crise Russo-ukrainienne sur le commerce extérieur et la pauvreté en Côte d’Ivoire
Alain NDIKUMANA
Alain NDIKUMANA
Alain NDIKUMANA
Impact of aid allocation in Africa: New sectoral and spatial evidence
Ngono Aristide Merlin
Ngono Aristide Merlin
Ngono Aristide Merlin
Résilience de la croissance économique en contexte pandémique: Une explication par les mesures de lutte contre la Covid-19
Aka Roger Kadjo
Aka Roger Kadjo
Aka Roger Kadjo
Estimation and asymptotic properties of a stationary univariate GARCH(p, q) process
Alain NDIKUMANA
Alain NDIKUMANA
Alain NDIKUMANA
Impact de l'allocation de l'aide en Afrique : Nouvelles preuves sectorielles et spatiales
Alain NDIKUMANA
Alain NDIKUMANA
Alain NDIKUMANA
Impact of aid allocation in Africa: New sectoral and spatial evidence
Ankouvi NAYO
Ankouvi NAYO
Ankouvi NAYO
Mutations culturales et dive

In [4]:
assert documents['doctorants'].extraction_method == 'structured_main_richer'
assert 'Alain NDIKUMANA' in documents['doctorants'].markdown
assert documents['doctorants'].extraction_diagnostics['selected_chars'] > 5000
assert documents['partenaires_recherche'].extraction_method == 'structured_main_richer'
assert 'INSD-Burkina Faso' in documents['partenaires_recherche'].markdown
assert documents['partenaires_recherche'].extraction_diagnostics['selected_chars'] > 4000
print('Extraction enrichie validée sur les deux pages récupérables.')
print('Les quatre autres pages restent signalées comme sources insuffisantes ou incohérentes.')


Extraction enrichie validée sur les deux pages récupérables.
Les quatre autres pages restent signalées comme sources insuffisantes ou incohérentes.
